# Blind Navigator — Training Notebook
**Dueling Double-DQN + PER + LSTM** | 16×16 grid | 6 maps | 5-phase curriculum

Designed as a production-quality RL foundation for assistive navigation.

In [1]:
import os, sys, time, json
import numpy as np

# Force reload core on re-run
for mod in list(sys.modules.keys()):
    if 'core' in mod:
        del sys.modules[mod]

sys.path.insert(0, '.')
os.makedirs('outputs', exist_ok=True)

from core import (
    BlindNavigatorEnv, DuelingPERAgent, QLearningAgent, SARSAAgent,
    run_episode, evaluate, train_curriculum,
    PHASE_CONFIG, PHASE_EPISODES, PHASE_EPSILON_RESTART,
    get_epsilon, HP, Action, ALL_MAPS, N_MAPS, smooth
)

print('✅ Imports OK')
print(f'   Grid: 16×16 | Maps: {N_MAPS} | OBS_DIM: {BlindNavigatorEnv.OBS_DIM}')
print(f'   Total training episodes: {sum(PHASE_EPISODES.values()):,}')

✅ Imports OK
   Grid: 16×16 | Maps: 6 | OBS_DIM: 20
   Total training episodes: 29,000


## Configuration

In [2]:
OUTPUT_DIR    = 'outputs'
SAVE_INTERVAL = 200   # print progress every N episodes per phase

# Which agents to train
# Set to ['Dueling-PER'] for fastest training, or include tabular agents for comparison
TRAIN_AGENTS = ['Dueling-PER', 'Q-Learning', 'SARSA']

print('Phase config:')
for ph, cfg in PHASE_CONFIG.items():
    print(f'  Phase {ph}: {PHASE_EPISODES[ph]:>5} eps | '
          f'ε₀={PHASE_EPSILON_RESTART[ph]:.2f} | '
          f'peds={cfg["n_pedestrians"]} | slip={cfg["slip_prob"]:.2f} | '
          f'maps={cfg["map_ids"]} | {cfg["description"]}')

Phase config:
  Phase 0:  3000 eps | ε₀=1.00 | peds=0 | slip=0.00 | maps=[0, 3] | Empty maps — learn basic navigation
  Phase 1:  4000 eps | ε₀=0.70 | peds=0 | slip=0.05 | maps=[0, 1, 2, 3] | Static hazards + edges + light slip
  Phase 2:  6000 eps | ε₀=0.60 | peds=2 | slip=0.08 | maps=[0, 1, 2, 3, 4] | Moving pedestrians — reactive avoidance
  Phase 3:  8000 eps | ε₀=0.40 | peds=4 | slip=0.10 | maps=[0, 1, 2, 3, 4, 5] | Full environment — all hazards, all maps
  Phase 4:  8000 eps | ε₀=0.30 | peds=5 | slip=0.12 | maps=[0, 1, 2, 3, 4, 5] | Transfer/generalization — max complexity


## Training with Checkpoint Support

In [3]:
def train_agent_with_checkpoints(agent, output_dir=OUTPUT_DIR, seed_off=0):
    """
    Curriculum training with:
    - Per-phase epsilon reset
    - Progress reporting every SAVE_INTERVAL episodes
    - Phase-level checkpointing
    - Resume from last completed phase
    """
    safe = agent.name.lower().replace(' ', '_').replace('-', '_')
    log_path = os.path.join(output_dir, f'{safe}_curriculum_logs.json')
    
    # Load existing logs
    logs = []
    if os.path.exists(log_path):
        with open(log_path) as f:
            logs = json.load(f)
    
    # Find completed phases
    completed_phases = set()
    for ph, n in PHASE_EPISODES.items():
        if sum(1 for l in logs if l['phase'] == ph) >= n:
            completed_phases.add(ph)
    
    if len(completed_phases) == len(PHASE_EPISODES):
        print(f'  ✓ {agent.name} already fully trained ({len(logs)} episodes)')
        return logs
    
    start_phase = max(completed_phases) + 1 if completed_phases else 0
    
    # Load latest checkpoint if resuming
    if start_phase > 0:
        prev_path = os.path.join(output_dir, f'{safe}_phase{start_phase-1}')
        try:
            agent.load(prev_path)
            print(f'  ↩ Loaded phase {start_phase-1} checkpoint')
        except:
            print(f'  ⚠ Could not load checkpoint, starting fresh')
        logs = [l for l in logs if l['phase'] in completed_phases]
    
    print(f'\n📦 Training {agent.name} from phase {start_phase}')
    total_episodes = sum(PHASE_EPISODES.values())
    
    for phase in range(start_phase, len(PHASE_EPISODES)):
        env = BlindNavigatorEnv(phase=phase)
        n_episodes = PHASE_EPISODES[phase]
        
        print(f'\n  Phase {phase} — {PHASE_CONFIG[phase]["description"]}')
        print(f'  Episodes: {n_episodes} | ε₀: {PHASE_EPSILON_RESTART[phase]:.2f} | '
              f'Maps: {PHASE_CONFIG[phase]["map_ids"]}')
        
        for ep in range(n_episodes):
            eps  = get_epsilon(ep, phase)
            seed = seed_off + len(logs)
            r, s, su, go, co = run_episode(env, agent, eps, seed)
            
            logs.append({
                'phase': phase, 'episode': ep, 'global_episode': len(logs),
                'total_reward': round(float(r), 3), 'steps': s,
                'success': bool(su), 'game_over': bool(go), 'collisions': int(co),
                'epsilon': round(float(eps), 5), 'seed': seed,
                'algorithm': agent.name, 'map_id': env.map_id,
            })
            
            if (ep + 1) % SAVE_INTERVAL == 0 or ep == n_episodes - 1:
                recent = logs[-SAVE_INTERVAL:]
                avg_r  = np.mean([l['total_reward'] for l in recent])
                sr     = np.mean([l['success']      for l in recent])
                pct    = len(logs) / total_episodes * 100
                print(f'    [{ep+1:>5}/{n_episodes}] '
                      f'SR={sr:.1%} | AvgR={avg_r:7.1f} | ε={eps:.3f} | '
                      f'Global: {pct:.1f}%')
                
                # Save logs periodically
                with open(log_path, 'w') as f:
                    json.dump(logs, f)
        
        # Save phase checkpoint
        phase_path = os.path.join(output_dir, f'{safe}_phase{phase}')
        agent.save(phase_path)
        print(f'  ✓ Phase {phase} checkpoint saved → {phase_path}')
    
    # Save final
    agent.save(os.path.join(output_dir, f'{safe}_final'))
    with open(log_path, 'w') as f:
        json.dump(logs, f)
    
    print(f'\n  ✓ {agent.name} complete — {len(logs)} episodes')
    return logs

print('✅ Training function ready')

✅ Training function ready


## Run Training

In [4]:
agent_configs = {
    'Dueling-PER': (DuelingPERAgent, {}, 0),
    'Q-Learning':  (QLearningAgent,  {}, 10000),
    'SARSA':       (SARSAAgent,       {}, 20000),
}

total_t = time.time()
results = {}

for name in TRAIN_AGENTS:
    if name not in agent_configs:
        print(f'Unknown agent: {name}')
        continue
    
    AgentClass, kw, seed_off = agent_configs[name]
    print(f'\n{'='*60}')
    print(f'  {name}')
    print(f'{'='*60}')
    
    t0 = time.time()
    try:
        agent = AgentClass(**kw) if kw else AgentClass()
        logs  = train_agent_with_checkpoints(agent, seed_off=seed_off)
        
        total_eps = sum(PHASE_EPISODES.values())
        if len(logs) >= total_eps:
            print(f'\n  Evaluating {name}...')
            ev = evaluate(agent, n_episodes=300, output_dir=OUTPUT_DIR)
            results[name] = {
                'success_rate':  ev['success_rate'],
                'mean_reward':   ev['mean_reward'],
                'collision_rate': ev['collision_rate'],
                'per_map':       ev['per_map'],
                'time_min':      round((time.time() - t0) / 60, 1),
            }
            print(f'  📊 SR={ev["success_rate"]:.1%} | '
                  f'R={ev["mean_reward"]:.1f} | '
                  f'Collisions={ev["collision_rate"]:.1%} | '
                  f'Time={results[name]["time_min"]}min')
            print(f'  Per-map SR: {ev["per_map"]}')
    except Exception as e:
        import traceback
        print(f'  ❌ Error: {e}')
        traceback.print_exc()

print(f'\n⏱ Total time: {(time.time()-total_t)/60:.1f} min')


  Dueling-PER

📦 Training Dueling-PER from phase 0

  Phase 0 — Empty maps — learn basic navigation
  Episodes: 3000 | ε₀: 1.00 | Maps: [0, 3]
    [  200/3000] SR=0.0% | AvgR= -137.7 | ε=0.937 | Global: 0.7%
    [  400/3000] SR=0.0% | AvgR= -109.2 | ε=0.874 | Global: 1.4%
    [  600/3000] SR=0.0% | AvgR= -105.8 | ε=0.810 | Global: 2.1%
    [  800/3000] SR=0.0% | AvgR=  -94.3 | ε=0.747 | Global: 2.8%
    [ 1000/3000] SR=1.5% | AvgR=  -43.8 | ε=0.684 | Global: 3.4%
    [ 1200/3000] SR=9.5% | AvgR=   10.4 | ε=0.620 | Global: 4.1%
    [ 1400/3000] SR=24.0% | AvgR=  107.2 | ε=0.557 | Global: 4.8%
    [ 1600/3000] SR=48.0% | AvgR=  246.2 | ε=0.494 | Global: 5.5%
    [ 1800/3000] SR=76.0% | AvgR=  417.8 | ε=0.430 | Global: 6.2%
    [ 2000/3000] SR=90.5% | AvgR=  506.0 | ε=0.367 | Global: 6.9%
    [ 2200/3000] SR=94.5% | AvgR=  531.9 | ε=0.304 | Global: 7.6%
    [ 2400/3000] SR=99.5% | AvgR=  576.2 | ε=0.240 | Global: 8.3%
    [ 2600/3000] SR=100.0% | AvgR=  586.8 | ε=0.177 | Global: 9.0%
   

## Results Summary

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
from core import load_logs, load_eval, smooth, COLORS_ALGO

if results:
    df = pd.DataFrame({
        k: {kk: vv for kk, vv in v.items() if kk != 'per_map'}
        for k, v in results.items()
    }).T
    print('\n📊 Final Results:')
    print(df.to_string())
else:
    print('No completed results yet.')


📊 Final Results:
             success_rate  mean_reward  collision_rate  time_min
Dueling-PER           1.0      545.413          0.2300     340.2
Q-Learning            0.0     -168.147          0.7367       9.1
SARSA                 0.0     -173.983          0.7467       8.7


In [6]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name in TRAIN_AGENTS:
    logs = load_logs(name, output_dir=OUTPUT_DIR)
    if not logs: continue
    color = COLORS_ALGO.get(name, '#888888')
    
    rewards   = [l['total_reward'] for l in logs]
    successes = [float(l['success']) for l in logs]
    
    axes[0].plot(smooth(rewards,   window=300), label=name, color=color, linewidth=1.5)
    axes[1].plot(smooth(successes, window=300), label=name, color=color, linewidth=1.5)

# Mark phase boundaries
boundaries = []
total = 0
for ph, n in PHASE_EPISODES.items():
    total += n
    boundaries.append((total, f'P{ph+1}'))

for ax in axes:
    for x, label in boundaries[:-1]:
        ax.axvline(x, color='gray', linestyle='--', alpha=0.4, linewidth=1)
        ax.text(x+50, ax.get_ylim()[1]*0.95, label, fontsize=8, color='gray')
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[0].set_title('Average Reward (smoothed, window=300)')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward')
axes[1].set_title('Success Rate (smoothed, window=300)')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Success Rate')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'learning_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Learning curves saved')

✅ Learning curves saved
